In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.titlesize"] = 13

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

TRADES_PATH = "historical_data.csv"
FEAR_GREED_PATH = "fear_greed_index.csv"

In [ ]:
trades = pd.read_csv(TRADES_PATH)
fg = pd.read_csv(FEAR_GREED_PATH)

print("Trades shape:", trades.shape)
print("Fear & Greed shape:", fg.shape)
trades.head()

In [ ]:
fg.head()

In [ ]:
trades.info()

In [ ]:
trades.columns = [c.strip().lower().replace(" ", "_") for c in trades.columns]
fg.columns = [c.strip().lower().replace(" ", "_") for c in fg.columns]

trades.rename(columns={"timestamp_ist": "trade_datetime"}, inplace=True)

trades["trade_datetime"] = pd.to_datetime(trades["trade_datetime"], format="%d-%m-%Y %H:%M")
trades["date"] = trades["trade_datetime"].dt.date.astype("datetime64[ns]")

fg["date"] = pd.to_datetime(fg["date"])

print("Trade date range:", trades["date"].min(), "->", trades["date"].max())
print("Sentiment date range:", fg["date"].min(), "->", fg["date"].max())

In [ ]:
print("Nulls per column (trades):")
print(trades.isna().sum()[trades.isna().sum() > 0])

dup_count = trades.duplicated().sum()
print(f"\nExact duplicate trade rows: {dup_count}")
trades = trades.drop_duplicates().reset_index(drop=True)

print("Nulls per column (fear/greed):")
print(fg.isna().sum())

print("\nAny non-positive execution prices?", (trades["execution_price"] <= 0).sum())
print("Any negative size_usd?", (trades["size_usd"] < 0).sum())

In [ ]:
min_fg, max_fg = fg["date"].min(), fg["date"].max()
before = len(trades)
trades = trades[(trades["date"] >= min_fg) & (trades["date"] <= max_fg)].copy()
print(f"Trades dropped outside sentiment coverage: {before - len(trades)} (kept {len(trades)})")

In [ ]:
trades["trade_hour"] = trades["trade_datetime"].dt.hour
trades["trade_dayofweek"] = trades["trade_datetime"].dt.day_name()
trades["is_weekend"] = trades["trade_datetime"].dt.dayofweek >= 5

trades["is_buy"] = trades["side"].str.upper().eq("BUY")
trades["is_win"] = trades["closed_pnl"] > 0
trades["is_loss"] = trades["closed_pnl"] < 0
trades["is_closing_trade"] = trades["closed_pnl"] != 0

trades["fee_pct"] = np.where(trades["size_usd"] > 0, trades["fee"] / trades["size_usd"] * 100, np.nan)
trades["pnl_net"] = trades["closed_pnl"] - trades["fee"]

trades["position_change_ratio"] = np.where(
    trades["start_position"].abs() > 0,
    trades["size_tokens"] / trades["start_position"].abs(),
    np.nan
)

trades["is_opening"] = trades["direction"].str.contains("Open", case=False, na=False)
trades["is_closing"] = trades["direction"].str.contains("Close", case=False, na=False)

trades[["trade_datetime","date","trade_hour","trade_dayofweek","is_weekend",
        "is_buy","is_win","is_loss","fee_pct","pnl_net","position_change_ratio"]].head()

In [ ]:
merged = trades.merge(fg[["date", "value", "classification"]], on="date", how="left")

def simplify(c):
    if c in ("Fear", "Extreme Fear"):
        return "Fear"
    if c in ("Greed", "Extreme Greed"):
        return "Greed"
    return "Neutral"

merged["sentiment_group"] = merged["classification"].apply(simplify)

print("Rows without a sentiment match:", merged["classification"].isna().sum())
merged[["date","classification","sentiment_group","value","closed_pnl","size_usd"]].head()

In [ ]:
fig, ax = plt.subplots(figsize=(11,3.5))
fg_window = fg[(fg["date"] >= trades["date"].min()) & (fg["date"] <= trades["date"].max())]
colors = fg_window["classification"].map({
    "Extreme Fear": "#7f1d1d", "Fear": "#dc2626", "Neutral": "#9ca3af",
    "Greed": "#16a34a", "Extreme Greed": "#166534"
})
ax.scatter(fg_window["date"], fg_window["value"], c=colors, s=8)
ax.set_title("Fear & Greed Index over the Trading Window")
ax.set_ylabel("Index value (0=Extreme Fear, 100=Extreme Greed)")
plt.tight_layout()
plt.show()

In [ ]:
order = ["Extreme Fear","Fear","Neutral","Greed","Extreme Greed"]
palette = {"Extreme Fear":"#7f1d1d","Fear":"#dc2626","Neutral":"#9ca3af","Greed":"#16a34a","Extreme Greed":"#166534"}

agg = merged.groupby("classification").agg(
    n_trades=("closed_pnl","size"),
    total_volume_usd=("size_usd","sum"),
    total_closed_pnl=("closed_pnl","sum"),
    avg_trade_size_usd=("size_usd","mean")
).reindex(order)

fig, axes = plt.subplots(1, 2, figsize=(13,4.5))
sns.barplot(x=agg.index, y=agg["n_trades"], hue=agg.index, palette=palette, legend=False, ax=axes[0])
axes[0].set_title("Number of Trades by Sentiment")
axes[0].set_xlabel(""); axes[0].set_ylabel("Trade count")
axes[0].tick_params(axis='x', rotation=20)

sns.barplot(x=agg.index, y=agg["total_volume_usd"]/1e6, hue=agg.index, palette=palette, legend=False, ax=axes[1])
axes[1].set_title("Total Trade Volume by Sentiment")
axes[1].set_xlabel(""); axes[1].set_ylabel("Volume (USD, millions)")
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

agg

In [ ]:
closing = merged[merged["is_closing_trade"]]

pnl_stats = closing.groupby("classification")["closed_pnl"].agg(
    trades="size", total_pnl="sum", mean_pnl="mean", median_pnl="median", win_rate=lambda s: (s>0).mean()
).reindex(order)
pnl_stats["win_rate"] = (pnl_stats["win_rate"]*100).round(2)
pnl_stats

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,4.5))

sns.barplot(x=pnl_stats.index, y=pnl_stats["total_pnl"]/1e6, hue=pnl_stats.index, palette=palette, legend=False, ax=axes[0])
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Total Closed PnL by Sentiment")
axes[0].set_ylabel("Total PnL (USD, millions)"); axes[0].set_xlabel("")
axes[0].tick_params(axis='x', rotation=20)

sns.barplot(x=pnl_stats.index, y=pnl_stats["win_rate"], hue=pnl_stats.index, palette=palette, legend=False, ax=axes[1])
axes[1].set_title("Win Rate (%) by Sentiment")
axes[1].set_ylabel("Win rate (%)"); axes[1].set_xlabel("")
axes[1].axhline(50, color="black", linestyle="--", linewidth=0.8)
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
plot_df = closing.copy()
lo, hi = plot_df["closed_pnl"].quantile([0.02, 0.98])
plot_df = plot_df[(plot_df["closed_pnl"] >= lo) & (plot_df["closed_pnl"] <= hi)]

fig, ax = plt.subplots(figsize=(11,5))
sns.boxplot(data=plot_df, x="classification", y="closed_pnl", order=order, hue="classification", palette=palette, legend=False, showfliers=False, ax=ax)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Closed PnL Distribution by Sentiment (2nd-98th percentile)")
ax.set_xlabel(""); ax.set_ylabel("Closed PnL (USD)")
plt.tight_layout()
plt.show()

In [ ]:
side_mix = pd.crosstab(merged["classification"], merged["side"], normalize="index").reindex(order) * 100

fig, ax = plt.subplots(figsize=(9,4.5))
side_mix.plot(kind="bar", stacked=True, color=["#2563eb","#f97316"], ax=ax)
ax.set_title("Buy vs. Sell Mix by Sentiment (%)")
ax.set_ylabel("% of trades"); ax.set_xlabel("")
ax.legend(title="Side")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

side_mix.round(1)

In [ ]:
top_coins_overall = merged["coin"].value_counts().head(8).index

coin_sent = (merged[merged["coin"].isin(top_coins_overall)]
             .groupby(["coin","sentiment_group"]).size()
             .unstack(fill_value=0))
coin_sent_pct = coin_sent.div(coin_sent.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(11,5))
coin_sent_pct[["Fear","Neutral","Greed"]].plot(kind="barh", stacked=True,
    color=["#dc2626","#9ca3af","#16a34a"], ax=ax)
ax.set_title("Trade Share by Sentiment Group for Top Traded Coins")
ax.set_xlabel("% of coin's trades"); ax.set_ylabel("")
ax.legend(title="Sentiment", bbox_to_anchor=(1.02,1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
daily = merged.groupby("date").agg(
    avg_size_usd=("size_usd","mean"),
    total_pnl=("closed_pnl","sum"),
    n_trades=("closed_pnl","size"),
    sentiment_value=("value","first")
).reset_index()

fig, ax1 = plt.subplots(figsize=(12,4.5))
ax1.plot(daily["date"], daily["avg_size_usd"], color="#2563eb", label="Avg trade size (USD)")
ax1.set_ylabel("Avg trade size (USD)", color="#2563eb")
ax1.tick_params(axis='y', labelcolor="#2563eb")

ax2 = ax1.twinx()
ax2.plot(daily["date"], daily["sentiment_value"], color="#dc2626", alpha=0.5, label="Sentiment (0-100)")
ax2.set_ylabel("Sentiment index", color="#dc2626")
ax2.tick_params(axis='y', labelcolor="#dc2626")

ax1.set_title("Daily Avg Trade Size vs. Market Sentiment Score")
plt.tight_layout()
plt.show()

corr = daily[["avg_size_usd","sentiment_value"]].corr().iloc[0,1]
print(f"Correlation (avg trade size vs sentiment score): {corr:.3f}")

In [ ]:
num_cols = ["execution_price","size_usd","closed_pnl","fee","fee_pct","value"]
corr_matrix = merged[num_cols].corr()

fig, ax = plt.subplots(figsize=(7,5.5))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Matrix — Trade & Sentiment Features")
plt.tight_layout()
plt.show()

In [ ]:
trader_stats = merged.groupby("account").agg(
    n_trades=("closed_pnl","size"),
    total_pnl=("closed_pnl","sum"),
    win_rate=("is_win","mean"),
    total_volume=("size_usd","sum"),
    avg_trade_size=("size_usd","mean"),
    total_fees=("fee","sum"),
).sort_values("total_pnl", ascending=False)

trader_stats["win_rate"] = (trader_stats["win_rate"]*100).round(2)
print(f"Total unique accounts: {trader_stats.shape[0]}")
trader_stats.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
top15 = trader_stats.head(15)
sns.barplot(x=top15["total_pnl"]/1000, y=[a[:10]+"..." for a in top15.index], hue=[a[:10]+"..." for a in top15.index], palette="crest", legend=False, ax=ax)
ax.set_title("Top 15 Traders by Total Closed PnL")
ax.set_xlabel("Total PnL (USD, thousands)"); ax.set_ylabel("Account")
plt.tight_layout()
plt.show()

In [ ]:
top20_accounts = trader_stats.head(20).index
best_trader_sent = (merged[merged["account"].isin(top20_accounts)]
                     .groupby("sentiment_group")["closed_pnl"].agg(["sum","mean","count"]))
best_trader_sent = best_trader_sent.reindex(["Fear","Neutral","Greed"])
print("Top-20 traders' PnL performance broken down by sentiment group:")
best_trader_sent

In [ ]:
bottom20_accounts = trader_stats.tail(20).index
worst_trader_sent = (merged[merged["account"].isin(bottom20_accounts)]
                      .groupby("sentiment_group")["closed_pnl"].agg(["sum","mean","count"]))
worst_trader_sent = worst_trader_sent.reindex(["Fear","Neutral","Greed"])
print("Bottom-20 traders' PnL performance broken down by sentiment group:")
worst_trader_sent

In [ ]:
fear_pnl = closing.loc[closing["sentiment_group"]=="Fear", "closed_pnl"]
greed_pnl = closing.loc[closing["sentiment_group"]=="Greed", "closed_pnl"]

t_stat, p_val = stats.ttest_ind(fear_pnl, greed_pnl, equal_var=False)
u_stat, p_val_mw = stats.mannwhitneyu(fear_pnl, greed_pnl, alternative="two-sided")

print(f"Fear  -> n={len(fear_pnl):,}  mean={fear_pnl.mean():.2f}  median={fear_pnl.median():.2f}")
print(f"Greed -> n={len(greed_pnl):,}  mean={greed_pnl.mean():.2f}  median={greed_pnl.median():.2f}")
print(f"\nWelch's t-test:      t = {t_stat:.3f}, p = {p_val:.4f}")
print(f"Mann-Whitney U test: U = {u_stat:.1f}, p = {p_val_mw:.4f}")

alpha = 0.05
sig = "statistically significant" if p_val < alpha else "not statistically significant"
print(f"\n=> The difference in mean PnL between Fear and Greed is {sig} at alpha={alpha}.")

In [ ]:
groups = [closing.loc[closing["classification"]==c, "closed_pnl"] for c in order]
f_stat, p_anova = stats.f_oneway(*groups)
print(f"One-way ANOVA across 5 sentiment classes: F = {f_stat:.3f}, p = {p_anova:.4f}")

In [ ]:
buy_ratio_by_sent = merged.groupby("classification")["is_buy"].mean().reindex(order) * 100
fig, ax = plt.subplots(figsize=(9,4.5))
sns.barplot(x=buy_ratio_by_sent.index, y=buy_ratio_by_sent.values, hue=buy_ratio_by_sent.index, palette=palette, legend=False, ax=ax)
ax.axhline(50, color="black", linestyle="--", linewidth=0.8)
ax.set_title("Share of BUY Orders by Sentiment (Contrarian vs. Momentum Signal)")
ax.set_ylabel("% BUY orders"); ax.set_xlabel("")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

buy_ratio_by_sent.round(2)

In [ ]:
hourly = merged.groupby(["trade_hour","sentiment_group"]).size().unstack(fill_value=0)
hourly_pct = hourly.div(hourly.sum(axis=0), axis=1) * 100

fig, ax = plt.subplots(figsize=(12,4.5))
hourly_pct[["Fear","Neutral","Greed"]].plot(ax=ax, marker="o", color=["#dc2626","#9ca3af","#16a34a"])
ax.set_title("Intraday Trading Pattern by Sentiment Group")
ax.set_xlabel("Hour of Day (IST)"); ax.set_ylabel("% of that group's trades")
ax.set_xticks(range(0,24,2))
plt.tight_layout()
plt.show()

In [ ]:
extreme_vs_mild = merged.copy()
extreme_vs_mild["extremity"] = extreme_vs_mild["classification"].apply(
    lambda c: "Extreme" if "Extreme" in str(c) else ("Neutral" if c=="Neutral" else "Mild")
)

extremity_stats = extreme_vs_mild[extreme_vs_mild["is_closing_trade"]].groupby("extremity")["closed_pnl"].agg(
    trades="size", mean_pnl="mean", std_pnl="std", win_rate=lambda s:(s>0).mean()*100
).reindex(["Mild","Extreme","Neutral"])
extremity_stats

In [ ]:
summary_lines = []

overall_fear_mean = fear_pnl.mean()
overall_greed_mean = greed_pnl.mean()
better = "Fear" if overall_fear_mean > overall_greed_mean else "Greed"
summary_lines.append(f"1. Average closed PnL per trade is higher during **{better}** regimes "
                      f"(Fear mean=${overall_fear_mean:.2f} vs Greed mean=${overall_greed_mean:.2f}).")

wr_fear = pnl_stats.loc["Fear","win_rate"] if "Fear" in pnl_stats.index else np.nan
wr_greed = pnl_stats.loc["Greed","win_rate"] if "Greed" in pnl_stats.index else np.nan
summary_lines.append(f"2. Win rate: Fear={wr_fear}% vs Greed={wr_greed}%.")

buy_fear = buy_ratio_by_sent.get("Fear", np.nan)
buy_greed = buy_ratio_by_sent.get("Greed", np.nan)
stance = "contrarian (net buying into Fear)" if buy_fear > 50 else "momentum-following (net selling into Fear)"
summary_lines.append(f"3. Traders in this dataset are broadly **{stance}**: {buy_fear:.1f}% of Fear-day orders are BUY "
                      f"vs {buy_greed:.1f}% on Greed days.")

summary_lines.append(f"4. Statistical test result: Fear vs Greed PnL difference is "
                      f"{'significant' if p_val < 0.05 else 'not significant'} (Welch t-test p={p_val:.4f}).")

top_pnl_group = pnl_stats["total_pnl"].idxmax()
summary_lines.append(f"5. In aggregate dollar terms, **{top_pnl_group}** generated the largest total closed PnL across all traders.")

summary_lines.append("6. The top-20 traders by total PnL should be studied as a 'smart money' cohort — "
                      "compare their sentiment-conditioned behaviour (section 6.1) against the bottom-20 cohort "
                      "to extract copyable strategy rules.")

print("KEY INSIGHTS\n" + "="*60)
for line in summary_lines:
    print(line, "\n")